In [18]:
%pip install sentence_transformers einops


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Prepare and describe the data

In [2]:
import json
with open("datas/train.jsonl", "r") as f:
    data_train = [json.loads(line) for line in f]


with open("datas/test.jsonl", "r") as f:
    data_test = [json.loads(line) for line in f]

with open("datas/devel.jsonl", "r") as f:
    data_devel = [json.loads(line) for line in f]

In [1]:
import pandas as pd
train_df = pd.DataFrame(data_train)
test_df = pd.DataFrame(data_test)

NameError: name 'data_train' is not defined

In [16]:
train_df["intent"]

0           calendar_set
1           calendar_set
2        audio_volume_up
3        iot_hue_lightup
4          weather_query
              ...       
11509     takeaway_query
11510     datetime_query
11511        email_query
11512     play_audiobook
11513       calendar_set
Name: intent, Length: 11514, dtype: str

In [17]:
train_df["intent"].value_counts()

intent
calendar_set      804
play_music        620
weather_query     559
calendar_query    558
general_quirky    546
                 ... 
events              1
convert             1
cleaning            1
settings            1
volume_other        1
Name: count, Length: 91, dtype: int64

In [27]:
df = pd.concat([train_df[["sentence", "intent"]],test_df[["sentence", "intent"]], pd.DataFrame(data_devel)[["sentence", "intent"]]], axis=0, ignore_index=True)

In [28]:
df = df.groupby("intent").filter(lambda x: len(x) >= 2)

In [29]:
df.describe(include="all")

,sentence,intent
count,16510,16510
unique,16486,82
top,do i have any new email,calendar_set
freq,3,1142


In [30]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

In [31]:
df

,sentence,intent
0,event,calendar_set
1,i need an event three days from now scheduled ...,calendar_set
2,turn up,audio_volume_up
3,brighten the lights a little bit,iot_hue_lightup
4,what's the current weather,weather_query
...,...,...
16516,can you give me local news on wayne county she...,news_query
16517,every light of room increase its intensity,iot_hue_lightup
16518,i would like some coffee now,iot_coffee
16519,what is the population of los angeles,qa_factoid


In [48]:
df = (
    df.groupby("intent", group_keys=False).filter(lambda x: len(x) >= 5).groupby("intent", group_keys=False).sample(n=5, random_state=42))

In [49]:
df["intent"].value_counts()

intent
alarm_query          5
alarm_remove         5
alarm_set            5
audio_volume_down    5
audio_volume_mute    5
                    ..
transport_query      5
transport_taxi       5
transport_ticket     5
transport_traffic    5
weather_query        5
Name: count, Length: 75, dtype: int64

In [7]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

NameError: name 'df' is not defined

In [9]:
import pandas as pd

In [24]:
df = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)

In [25]:
df["intent"].value_counts()

intent
alarm_query                 12
alarm_remove                12
alarm_set                   12
audio_volume_down           12
audio_volume_mute           12
audio_volume_up             12
calendar_query              12
calendar_remove             12
calendar_set                12
cooking_query               12
cooking_recipe              12
datetime_convert            12
datetime_query              12
email_addcontact            12
email_query                 12
email_querycontact          12
email_sendemail             12
general_greet               12
general_joke                12
iot_cleaning                12
iot_coffee                  12
iot_hue_lightchange         12
iot_hue_lightoff            12
iot_hue_lighton             12
iot_wemo_off                12
iot_wemo_on                 12
lists_createoradd           12
lists_query                 12
lists_remove                12
music_query                 12
news_query                  12
play_audiobook              12
p

## Pipeline

In [1]:
import pandas as pd
df = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)
import numpy as np

def prepare_model(model, dataset=None):
    if dataset is None:
        dataset = df
    intents = sorted(dataset["intent"].unique())

    example_vectors = model.encode(
        dataset["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    action_vectors = np.stack([
        example_vectors[dataset["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents
    ])
    action_vectors /= np.linalg.norm(action_vectors, axis=1, keepdims=True)
    return action_vectors

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
THRESHOLD = 0.6
import numpy as np
df_test = pd.read_json("datas/df_test.jsonl", orient="records", lines=True)
df_train = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)
intents = sorted(df_train["intent"].unique())

def test_model(model, action_vectors_for_model, dataset=None, class_names=None, threshold=None):
    if dataset is None:
        dataset = df_test
    if class_names is None:
        class_names = intents
    if threshold is None:
        threshold = THRESHOLD
    test_vectors = model.encode(
        dataset["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=True
    )
    similarities = test_vectors @ action_vectors_for_model.T
    best_indices = np.argmax(similarities, axis=1)
    best_scores = similarities.max(axis=1)
    y_pred = np.asarray(class_names, dtype=object)[best_indices]
    y_pred[best_scores < threshold] = "unknown"
    y_true = dataset["intent"].to_numpy()

    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1_macro = f1_score(y_true, y_pred, labels=class_names, average="macro", zero_division=0)
    metrics = {"accuracy": test_accuracy, "f1_macro": test_f1_macro}
    print(f"Test accuracy: {test_accuracy}")
    print(f"Test macro F1: {test_f1_macro}")



    for offset, language in enumerate(("Russian", "Kazakh", "English")):
        mask = np.arange(len(dataset)) % 3 == offset
        language_accuracy = accuracy_score(y_true[mask], y_pred[mask])
        print(f"{language} accuracy: {language_accuracy:.4f}")
        metrics[f"accuracy_{language.lower()}"] = language_accuracy

    return metrics


In [3]:
def test_pipe(model, dataset=None, train_dataset=None, threshold=None):
    if train_dataset is None:
        train_dataset = df_train
    class_names = sorted(train_dataset["intent"].unique())
    action_vectors_for_model = prepare_model(model, dataset=train_dataset)
    return test_model(
        model, action_vectors_for_model, dataset=dataset,
        class_names=class_names, threshold=threshold,
    )


In [4]:
import pandas as pd

df_small = pd.read_json("datas/df_small.jsonl", orient="records", lines=True)
df_small_train = pd.read_json("datas/df_small_train.jsonl", orient="records", lines=True)
df_small_test = pd.read_json("datas/df_small_test.jsonl", orient="records", lines=True)
print(f"Small dataset: {len(df_small)} examples, "
      f"{len(df_small_train)} train, {len(df_small_test)} test, "
      f"{df_small['intent'].nunique()} intents")


Small dataset: 60 examples, 48 train, 12 test, 4 intents


In [5]:
from sentence_transformers import SentenceTransformer

In [6]:
import numpy as np

model_names = [
    "BAAI/bge-m3",
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "Hiveurban/multilingual-e5-large-pooled",
    "Darmm/darmm-embed-kazakh-v2",
    "intfloat/multilingual-e5-large-instruct",
    "Qwen/Qwen3-Embedding-0.6B",
]
model_desc = [
    {"params_m": 568, "dimension": 1024, "max_input": 8192},
    {"params_m": 118, "dimension": 384,  "max_input": 512},
    {"params_m": 278, "dimension": 768,  "max_input": 512},
    {"params_m": 560, "dimension": 1024, "max_input": 512},
    {"params_m": 560, "dimension": 1024, "max_input": 512},
    {"params_m": 568, "dimension": 1024, "max_input": 8192},
    {"params_m": 560, "dimension": 1024, "max_input": 512},
    {"params_m": 600, "dimension": 1024, "max_input": 32768},
]


In [ ]:

import gc

import torch


results = []

for model_name, desc in zip(model_names, model_desc):
    print(model_name)
    model = SentenceTransformer(model_name)
    metrics = test_pipe(model=model, dataset=df_test)
    results.append({"model": model_name, **metrics, **desc})
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df


In [11]:
results_error = []
import gc
import torch
df_test_error = pd.read_json("datas/df_test_error.jsonl", orient="records", lines=True)
for model_name, desc in zip(model_names, model_desc):
    print(model_name)
    model = SentenceTransformer(model_name)
    metrics = test_pipe(model=model, dataset=df_test_error)
    results_error.append({"model": model_name, **metrics, **desc})
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_error_df = pd.DataFrame(results_error)
results_error_df


BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Test accuracy: 0.7066666666666667
Test macro F1: 0.6895873015873015
Russian accuracy: 0.7600
Kazakh accuracy: 0.6000
English accuracy: 0.7600
intfloat/multilingual-e5-small


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Test accuracy: 0.7466666666666667
Test macro F1: 0.7256031746031745
Russian accuracy: 0.8200
Kazakh accuracy: 0.6200
English accuracy: 0.8000
Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Test accuracy: 0.5066666666666667
Test macro F1: 0.5634761904761905
Russian accuracy: 0.5600
Kazakh accuracy: 0.2800
English accuracy: 0.6800


,model,accuracy,f1_macro,accuracy_russian,accuracy_kazakh,accuracy_english,params_m,dimension,max_input
0,BAAI/bge-m3,0.706667,0.689587,0.76,0.60,0.76,568,1024,8192
1,intfloat/multilingual-e5-small,0.460000,0.434302,0.54,0.40,0.44,118,384,512
2,intfloat/multilingual-e5-base,0.613333,0.595544,0.70,0.50,0.64,278,768,512
3,intfloat/multilingual-e5-large,0.720000,0.704052,0.78,0.64,0.74,560,1024,512
4,Hiveurban/multilingual-e5-large-pooled,0.720000,0.704052,0.78,0.64,0.74,560,1024,512
5,Darmm/darmm-embed-kazakh-v2,0.433333,0.430286,0.44,0.40,0.46,568,1024,8192
6,intfloat/multilingual-e5-large-instruct,0.746667,0.725603,0.82,0.62,0.80,560,1024,512
7,Qwen/Qwen3-Embedding-0.6B,0.506667,0.563476,0.56,0.28,0.68,600,1024,32768


In [13]:
results_error_df.sort_values(by="f1_macro", ascending=False, inplace=True)

In [15]:
results_error_df.to_csv("results_error.csv", index=False)

In [18]:
results_df = pd.read_csv("results.csv")
results_df

,model,accuracy,f1_macro,accuracy_russian,accuracy_kazakh,accuracy_english,params_m,dimension,max_input
0,BAAI/bge-m3,0.820000,0.808286,0.82,0.78,0.86,568,1024,8192
1,intfloat/multilingual-e5-large-instruct,0.800000,0.781846,0.88,0.68,0.84,560,1024,512
2,intfloat/multilingual-e5-large,0.746667,0.738004,0.78,0.66,0.80,560,1024,512
3,Hiveurban/multilingual-e5-large-pooled,0.746667,0.738004,0.78,0.66,0.80,560,1024,512
4,intfloat/multilingual-e5-base,0.700000,0.666841,0.76,0.64,0.70,278,768,512
5,Qwen/Qwen3-Embedding-0.6B,0.566667,0.623952,0.64,0.30,0.76,600,1024,32768
6,intfloat/multilingual-e5-small,0.560000,0.542004,0.58,0.48,0.62,118,384,512
7,Darmm/darmm-embed-kazakh-v2,0.506667,0.505766,0.50,0.46,0.56,568,1024,8192


In [29]:
results_df[["accuracy", "f1_macro", 'accuracy_russian', 'accuracy_kazakh', 'accuracy_english']].iloc[0] - results_error_df[["accuracy", "f1_macro", 'accuracy_russian', 'accuracy_kazakh', 'accuracy_english']].iloc[0]

accuracy            0.073333
f1_macro            0.082683
accuracy_russian    0.000000
accuracy_kazakh     0.160000
accuracy_english    0.060000
dtype: float64

In [ ]:
model = SentenceTransformer(model_names[0])
test_pipe(model=model, dataset=df_small_test, train_dataset=df_small_train, threshold=0.40)

## Fast APi

In [46]:
def predict(query, model):
    action_vectors_for_model = prepare_model(model)

    query_vector = model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    )[0]
    similarities = action_vectors_for_model @ query_vector

    top_indices = np.argsort(similarities)[-3:][::-1]
    return [
        {
            "endpoint": intents[index],
            "confidence": float(np.clip(similarities[index], 0, 1)),
        }
        for index in top_indices
    ]
df_small_train = pd.read_json("datas/df_small_train.jsonl", orient="records", lines=True)
df_small_test = pd.read_json("datas/df_small_test.jsonl", orient="records", lines=True)

def predict_small(query, model):
    intents_small = sorted(df_small_train["intent"].unique())
    train_vectors = model.encode(df_small_train["sentence"].tolist(), normalize_embeddings=True)
    centers = np.stack([
        train_vectors[df_small_train["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents_small
    ])
    centers /= np.linalg.norm(centers, axis=1, keepdims=True)

    query_vector = model.encode([query], normalize_embeddings=True)[0]
    scores = centers @ query_vector
    top_indices = np.argsort(scores)[-3:][::-1]
    return [
        {
            "endpoint": intents_small[index],
            "confidence": float(np.clip(scores[index], 0, 1)),
        }
        for index in top_indices
    ]

In [47]:
from fastapi import FastAPI

from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
import time
model = SentenceTransformer("BAAI/bge-m3")
app = FastAPI()
class QueryRequest(BaseModel):
    query: str = Field(min_length=1)


@app.post("/classify")
def classify(request: QueryRequest):
    start_time = time.perf_counter()

    result = predict(request.query, model)
    for item in result:
        item["threshold_passed"] = item["confidence"] >= THRESHOLD
        item["confidence"] = round(item["confidence"], 4)

    print(f"/classify: {time.perf_counter() - start_time:.3f}s")
    return result


@app.post("/prod")
def prod(request: QueryRequest):
    start_time = time.perf_counter()

    result = predict_small(request.query, model)
    for item in result:
        item["threshold_passed"] = item["confidence"] >= 0.40
        item["confidence"] = round(item["confidence"], 4)

    print(f"/prod: {time.perf_counter() - start_time:.3f}s")
    return result


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [48]:
async def serve():
    import uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000))
    await server.serve()

In [49]:
await serve()

INFO:     Started server process [47358]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


/classify: 0.815s
INFO:     127.0.0.1:58696 - "POST /classify HTTP/1.1" 200 OK
/prod: 0.185s
INFO:     127.0.0.1:43578 - "POST /prod HTTP/1.1" 200 OK
/prod: 0.183s
INFO:     127.0.0.1:60592 - "POST /prod HTTP/1.1" 200 OK
/prod: 0.167s
INFO:     127.0.0.1:53980 - "POST /prod HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [47358]
